In [1]:
import folium
from folium.plugins import MarkerCluster

In [2]:
import numpy as np

size = 100
lons = np.random.randint(-180, 180, size=size)
lats = np.random.randint(-90, 90, size=size)
dummy = np.random.randint(0, 20, size=size)

locations = list(zip(lats, lons))
popups = ["lon:{}<br>lat:{}".format(lon, lat) for (lat, lon) in locations]

Adding all icons in a single call

In [3]:
icon_create_function = """\
function(cluster) {
    return L.divIcon({
    html: '<b>' + cluster.getChildCount() + '</b>',
    className: 'marker-cluster marker-cluster-large',
    iconSize: new L.Point(20, 20)
    });
}"""

Explicit loop allow for customization in the loop.

In [4]:
%%time

m = folium.Map(
    location=[np.mean(lats), np.mean(lons)],
    tiles='Cartodb Positron',
    zoom_start=1
)

marker_cluster = MarkerCluster(
    name='1000 clustered icons',
    overlay=True,
    control=False,
    icon_create_function=None
)

for k in range(size):
    location = lats[k], lons[k]
    marker = folium.Marker(location=location)
    popup = 'lon:{}<br>lat:{}'.format(location[1], location[0])
    folium.Popup(popup).add_to(marker)
    marker_cluster.add_child(marker)

marker_cluster.add_to(m)

folium.LayerControl().add_to(m);

Wall time: 15 ms


In [5]:
m

In [6]:
marker_data =[]
for lng, lat, label in zip(lons, lats, dummy):
    marker_data.append({'location':[lat,lng],'population':round(label)})

marker_data=tuple(marker_data)
marker_data

({'location': [76, -106], 'population': 18},
 {'location': [13, -18], 'population': 6},
 {'location': [-60, 91], 'population': 1},
 {'location': [-80, -59], 'population': 11},
 {'location': [-65, 70], 'population': 13},
 {'location': [33, -114], 'population': 16},
 {'location': [28, 142], 'population': 8},
 {'location': [-64, -68], 'population': 13},
 {'location': [-14, 119], 'population': 14},
 {'location': [-52, -71], 'population': 14},
 {'location': [-75, -96], 'population': 8},
 {'location': [-55, 26], 'population': 10},
 {'location': [-10, 106], 'population': 19},
 {'location': [-46, -44], 'population': 17},
 {'location': [-11, -104], 'population': 0},
 {'location': [4, 6], 'population': 17},
 {'location': [86, -63], 'population': 13},
 {'location': [-18, -178], 'population': 8},
 {'location': [-24, -94], 'population': 18},
 {'location': [-85, -22], 'population': 11},
 {'location': [38, -137], 'population': 4},
 {'location': [-58, 173], 'population': 18},
 {'location': [31, 67], '

In [7]:
import json
import folium
from folium import Marker
from folium.plugins import MarkerCluster
from jinja2 import Template


class MarkerWithProps(Marker):
    _template = Template(u"""
        {% macro script(this, kwargs) %}
        var {{this.get_name()}} = L.marker(
            [{{this.location[0]}}, {{this.location[1]}}],
            {
                icon: new L.Icon.Default(),
                {%- if this.draggable %}
                draggable: true,
                autoPan: true,
                {%- endif %}
                {%- if this.props %}
                props : {{ this.props }} 
                {%- endif %}
                }
            )
            .addTo({{this._parent.get_name()}});
        {% endmacro %}
        """)
    def __init__(self, location, popup=None, tooltip=None, icon=None, draggable=False, props = None ):
        super(MarkerWithProps, self).__init__(location=location,popup=popup,tooltip=tooltip,icon=icon,draggable=draggable)
        self.props = json.loads(json.dumps(props)) 
        
icon_create_function = '''
    function(cluster) {
        var markers = cluster.getAllChildMarkers();
        var sum = 0;
        for (var i = 0; i < markers.length; i++) {
            sum += markers[i].options.props.population;
        }
        var avg = Math.round(sum/cluster.getChildCount());
        
        var c = ' marker-cluster-';

        if (avg < 6) {
            c += 'small';
        } else if (avg < 12) {
            c += 'medium';
        } else {
            c += 'large';
        }

        return L.divIcon({
             html: '<div><span>' + avg + '</span></div>',
             className: 'marker-cluster marker-cluster' + c,
             iconSize: new L.Point(40, 40)
        });
    }
'''


marker_cluster = MarkerCluster(icon_create_function=icon_create_function)

for marker_item in marker_data:
    marker = MarkerWithProps(
        location=marker_item['location'],
        props = { 'population': marker_item['population']},
        icon=folium.Icon(color='red', icon='building-sign',prefix = 'glyphicon')
    )
    marker.add_to(marker_cluster)

In [8]:
m = folium.Map(
    location=[np.mean(lats), np.mean(lons)],
    tiles='Cartodb Positron',
    zoom_start=1
)
marker_cluster.add_to(m) 

In [9]:
m